In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_sherpa_module import *
from association_module import *
from morbidity_chimere_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_sherpa(sc="s1", pol="ug_NO2", year=2019)
# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))
# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())
# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Concentrations in 2019 and 2019 are calculated for the pollutant 'ug_no2' (s1).
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod    comname depcod   depname  regcod    regname  \
0  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
1  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
2  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
3  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   
4  670430101   Annexe  67043  Bischheim     67  Bas-Rhin    44.0  Grand Est   

   age   pop2019   pop2030   pop2050  mort2019  mort2030  mort2050  \
0    0  1.001248  1.028860  1.007603  0.003590  0.003423  0.002972   
1    1  1.024924  1.022945  1.015467  0.000575  0.000556  0.000491   
2    2  1.049225  1.018921  1.023012  0.000261  0.000246  0.000219   
3    3  1.078030  1.016966  1.029914  0.000157  0.000155  0.000142   
4    4  1.114215  1.017214  1.035872  0.000121  0.000118  0.000109   

                                            geometry  
0  POLYGON ((1052346.7 6848413, 1052502.3 6848474

In [4]:
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union
# ---- A) Merge IRIS-level shapefiles ----
donnees_merged = gpd.GeoDataFrame(
    pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True),
    crs=donnees_shp_1.crs
)
donnees_merged = donnees_merged.copy()
donnees_merged["comcod"] = donnees_merged["comcod"].astype(str).str.zfill(5).str.strip()
donnees_merged["geometry"] = donnees_merged["geometry"].apply(
    lambda g: g.buffer(0) if g is not None and not g.is_valid else g
)

# ---- B) Aggregate to commune level ----
agg_dict = {
    "geometry": ("geometry", lambda x: unary_union(
        [g.buffer(0) for g in x if g is not None and g.is_valid]
    ))
}
if "pop2019" in donnees_merged.columns:
    agg_dict["pop2019"] = ("pop2019", "sum")
if "comname" in donnees_merged.columns:
    agg_dict["comname"] = ("comname", "first")

commune = (
    donnees_merged.groupby("comcod")
    .agg(**agg_dict)
    .reset_index()
)
commune = gpd.GeoDataFrame(commune, geometry="geometry", crs=donnees_merged.crs)

# ---- C) Load and clean INSEE density file ----
density_data = pd.read_excel("data/1-processed-data/grille_densite_niveau_detaille_2022.xlsx")
def normalize_str(s):
    """Uppercase, strip accents, remove punctuation for fuzzy matching."""
    import unicodedata, re
    s = str(s).upper().strip()
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")  # strip accents
    s = re.sub(r"[^A-Z0-9]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

density_data["CODGEO"] = density_data["CODGEO"].astype(str).str.zfill(5).str.strip()
density_data["LIBGEO_norm"] = density_data["LIBGEO"].apply(normalize_str)

# Pass 1: exact code match
commune = commune.merge(
    density_data[["CODGEO", "LIBGEO", "DENS"]],
    left_on="comcod", right_on="CODGEO",
    how="left"
)

# Pass 2: name match for still-unmatched rows
if "comname" in commune.columns:
    commune["comname_norm"] = commune["comname"].apply(normalize_str)
    unmatched_mask = commune["DENS"].isna()
    if unmatched_mask.any():
        name_lut = density_data[["LIBGEO_norm", "DENS"]].drop_duplicates("LIBGEO_norm")
        commune.loc[unmatched_mask, "DENS"] = (
            commune.loc[unmatched_mask, "comname_norm"]
            .map(name_lut.set_index("LIBGEO_norm")["DENS"])
        )

# Pass 3: department-level fallback for remaining NAs
#   Communes that merged/split since 2022 won't match by code or name.
#   Fall back to the most common DENS value in the same department (first 2 digits).
if commune["DENS"].isna().any():
    commune["dep"] = commune["comcod"].str[:2]
    dept_mode = (
        commune.dropna(subset=["DENS"])
        .groupby("dep")["DENS"]
        .agg(lambda x: x.mode().iloc[0])
    )
    still_na = commune["DENS"].isna()
    commune.loc[still_na, "DENS"] = commune.loc[still_na, "dep"].map(dept_mode)
    n_fallback = still_na.sum()
    if n_fallback > 0:
        print(f"[FALLBACK] {n_fallback} communes filled by department mode")

# Diagnostic: how many are still NA after all passes?
n_na = commune["DENS"].isna().sum()
print(f"[COVERAGE] Communes still NA after all passes: {n_na}")
if n_na > 0:
    print(commune[commune["DENS"].isna()][["comcod", "comname"] if "comname" in commune.columns else ["comcod"]].head(20))

# ---- D) Classify ----
def classify_cluster(dens):
    """
    INSEE (4 modalités):
      1 : commune densément peuplée
      2 : commune de densité intermédiaire
      3 : commune peu dense
      4 : commune très peu dense
    """
    if pd.isna(dens):
        return "NA"
    d = int(dens)
    if d == 1:
        return "Urban"          # densément peuplée
    elif d == 2:
        return "Peri-Urban"     # densité intermédiaire
    elif d in (3, 4):
        return "Rural"          # peu dense / très peu dense
    else:
        return "Urban"

commune["area_cluster"] = commune["DENS"].apply(classify_cluster)
print("\n[COMMUNE] area_cluster distribution:")
print(commune["area_cluster"].value_counts(dropna=False))

# ---- E) Push cluster back to IRIS level (donnees_merged) ----
if "area_cluster" in donnees_merged.columns:
    donnees_merged = donnees_merged.drop(columns=["area_cluster"])

donnees_merged = donnees_merged.merge(
    commune[["comcod", "area_cluster"]],
    on="comcod", how="left"
)

donnees_merged["age"] = pd.to_numeric(donnees_merged["age"], errors="coerce")
donnees_merged_new = donnees_merged.loc[
    donnees_merged["age"].between(30, 99, inclusive="both")
].copy()

print("\n[IRIS] area_cluster distribution (ages 30–99):")
print(donnees_merged_new["area_cluster"].value_counts(dropna=False))

# ---- F) Push cluster to donnees_exportees_transformed ----
cluster_lut = (
    donnees_merged_new[["comcod", "area_cluster"]]
    .drop_duplicates()
    .copy()
)
cluster_lut["comcod"] = cluster_lut["comcod"].astype(str).str.zfill(5).str.strip()

donnees_exportees_transformed["comcod"] = (
    donnees_exportees_transformed["comcod"].astype(str).str.zfill(5).str.strip()
)
if "area_cluster" in donnees_exportees_transformed.columns:
    donnees_exportees_transformed = donnees_exportees_transformed.drop(columns=["area_cluster"])

donnees_exportees_transformed = donnees_exportees_transformed.merge(
    cluster_lut, on="comcod", how="left"
)
print("\n[TRANSFORMED] area_cluster distribution:")
print(donnees_exportees_transformed["area_cluster"].value_counts(dropna=False))

[FALLBACK] 42 communes filled by department mode
[COVERAGE] Communes still NA after all passes: 0

[COMMUNE] area_cluster distribution:
area_cluster
Rural         29693
Peri-Urban     3326
Urban           776
Name: count, dtype: int64

[IRIS] area_cluster distribution (ages 30–99):
area_cluster
Rural         2108330
Urban          656810
Peri-Urban     546700
Name: count, dtype: int64

[TRANSFORMED] area_cluster distribution:
area_cluster
Rural         30119
Urban          9383
Peri-Urban     7810
Name: count, dtype: int64


In [5]:
# ----------------------------
# Life Table Function
# ----------------------------
def life_expectancy_table(age, m_x, interval=1, a_x=0.5, radix=100000):
    """
    Calculate life expectancy at each age using a standard abridged life table.

    Parameters:
        age (array-like): Ages.
        m_x (array-like): Mortality rates for each age.
        interval (int or float): Age interval length (default 1).
        a_x (float or array-like): Fraction of interval lived by those who die.
            Default is 0.5.
        radix (int or float): Starting population (default 100,000; scaling factor only).

    Returns:
        np.ndarray: Life expectancy at each age (e_x).
    """
    age = np.asarray(age)
    n = len(age)

    if a_x is None:
        a_x = np.full(n, 0.5, dtype=float)
    elif np.isscalar(a_x):
        a_x = np.full(n, float(a_x), dtype=float)
    else:
        a_x = np.asarray(a_x, dtype=float)
        if len(a_x) != n:
            raise ValueError("`a_x` must be a scalar or have the same length as `age`.")

    m_x = np.asarray(
        np.nan_to_num(m_x, nan=0.0, posinf=np.finfo(float).max, neginf=0.0),
        dtype=float
    )
    m_x[m_x < 0] = 0.0

    q_x = (interval * m_x) / (1.0 + (interval - a_x) * m_x)
    q_x = np.clip(q_x, 0.0, 1.0)

    l_x = np.zeros(n, dtype=float)
    l_x[0] = float(radix)
    for i in range(1, n):
        l_x[i] = l_x[i - 1] * (1.0 - q_x[i - 1])

    L_x = np.zeros(n, dtype=float)
    for i in range(n - 1):
        L_x[i] = interval * (l_x[i + 1] + a_x[i] * (l_x[i] - l_x[i + 1]))

    # Open-ended last age interval
    if m_x[-1] > 0:
        L_x[-1] = l_x[-1] / m_x[-1]
    else:
        L_x[-1] = interval * l_x[-1]

    T_x = np.zeros(n, dtype=float)
    T_x[-1] = L_x[-1]
    for i in range(n - 2, -1, -1):
        T_x[i] = T_x[i + 1] + L_x[i]

    e_x = T_x / np.where(l_x > 0, l_x, np.nan)
    return e_x

def mortalite_age_commune_monte_carlo(
    mort_comm_age, donnees_expo, donnees_transformed, annee, pol, num_simulations=1000, seed=None
):
    import numpy as np
    import pandas as pd

    if seed is not None:
        np.random.seed(seed)

    mort_comm_age       = mort_comm_age.copy()
    donnees_expo        = donnees_expo.copy()
    donnees_transformed = donnees_transformed.copy()

    def clean_iris(s):
        return s.astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True).str.zfill(9)
    def clean_com(s):
        return s.astype(str).str.strip().str.upper().str.replace(r"\.0$", "", regex=True).str.zfill(5)

    mort_comm_age["iriscod"] = clean_iris(mort_comm_age["iriscod"])
    mort_comm_age["comcod"]  = clean_com(mort_comm_age["comcod"])
    donnees_expo["iriscod"]        = clean_iris(donnees_expo["iriscod"])
    donnees_transformed["iriscod"] = clean_iris(donnees_transformed["iriscod"])
    donnees_transformed["comcod"]  = clean_com(donnees_transformed["comcod"])

    # --- DEDUPLICATION: enforce one row per (iriscod, age) ---
    before = len(mort_comm_age)
    mort_comm_age = mort_comm_age.drop_duplicates(subset=["iriscod", "age"]).reset_index(drop=True)
    after = len(mort_comm_age)
    if before != after:
        print(f"[DEDUP] mort_comm_age: {before:,} → {after:,} rows ({before - after:,} duplicates removed)")

    donnees_expo = donnees_expo.merge(
        donnees_transformed[["iriscod", "comcod"]].drop_duplicates(),
        on="iriscod", how="left"
    )
    n_null = donnees_expo["comcod"].isna().sum()
    if n_null > 0:
        print(f"WARNING: {n_null} IRISes in expo not found in donnees_transformed")
    donnees_expo["comcod"] = donnees_expo["comcod"].fillna(donnees_expo["iriscod"].str[:5])

    mort_comm_age["age"]          = pd.to_numeric(mort_comm_age["age"],          errors="coerce")
    mort_comm_age[f"pop{annee}"]  = pd.to_numeric(mort_comm_age[f"pop{annee}"],  errors="coerce")
    mort_comm_age[f"mort{annee}"] = pd.to_numeric(mort_comm_age[f"mort{annee}"], errors="coerce")
    donnees_expo["meandelta"]     = pd.to_numeric(donnees_expo["meandelta"],      errors="coerce")

    mort_comm_age = mort_comm_age.dropna(subset=["age", f"pop{annee}", f"mort{annee}"])
    mort_comm_age = mort_comm_age[(mort_comm_age["age"] >= 30) & (mort_comm_age["age"] <= 99)].copy()
    mort_comm_age["pop_age"]   = mort_comm_age[f"pop{annee}"].clip(lower=0)
    mort_comm_age["morta_age"] = mort_comm_age[f"mort{annee}"].clip(lower=0)

    # --- Step 1: broadcast delta to every IRIS×age row ---
    mort_comm_age = mort_comm_age.merge(
        donnees_expo[["iriscod", "comcod", "meandelta"]].drop_duplicates("iriscod"),
        on="iriscod", how="left", suffixes=("", "_expo")
    )
    mort_comm_age["comcod"] = mort_comm_age["comcod_expo"].combine_first(mort_comm_age["comcod"])
    mort_comm_age = mort_comm_age.drop(columns=["comcod_expo"], errors="ignore")
    mort_comm_age["meandelta"] = mort_comm_age["meandelta"].fillna(0)

    # --- Step 2: commune×age aggregation ---
    mort_comm_age["pop_x_delta"] = mort_comm_age["pop_age"] * mort_comm_age["meandelta"]

    comm_age = (
        mort_comm_age.groupby(["comcod", "age"], as_index=False)
        .agg(
            pop_age     = ("pop_age",     "sum"),
            morta_age   = ("morta_age",   "sum"),
            pop_x_delta = ("pop_x_delta", "sum"),
        )
    )
    comm_age["meandelta_comm"] = np.where(
        comm_age["pop_age"] > 0,
        comm_age["pop_x_delta"] / comm_age["pop_age"],
        0.0
    )
    comm_age = comm_age.drop(columns=["pop_x_delta"])
    comm_age["comcod"] = clean_com(comm_age["comcod"].astype(str))

    # --- Step 3: Coverage reporting ---
    pop_total    = comm_age["pop_age"].sum()
    mort_total   = comm_age["morta_age"].sum()
    pop_covered  = comm_age.loc[comm_age["meandelta_comm"] > 0,   "pop_age"].sum()
    pop_zero     = comm_age.loc[comm_age["meandelta_comm"] == 0,  "pop_age"].sum()
    pop_excluded = comm_age.loc[comm_age["meandelta_comm"].isna(), "pop_age"].sum()
    mort_covered = comm_age.loc[comm_age["meandelta_comm"] > 0,   "morta_age"].sum()
    mort_excluded= comm_age.loc[comm_age["meandelta_comm"].isna(), "morta_age"].sum()

    n_comm_covered = comm_age.loc[comm_age["meandelta_comm"] > 0,   "comcod"].nunique()
    n_comm_zero    = comm_age.loc[comm_age["meandelta_comm"] == 0,  "comcod"].nunique()
    n_comm_na      = comm_age.loc[comm_age["meandelta_comm"].isna(),"comcod"].nunique()

    print(f"[COVERAGE] Communes with delta > 0   : {n_comm_covered}")
    print(f"[COVERAGE] Communes with delta = 0   : {n_comm_zero}")
    print(f"[COVERAGE] Communes with delta NaN   : {n_comm_na} (excluded)")
    print(f"\n[COVERAGE] Population with delta > 0 : {pop_covered:>12,.0f} ({pop_covered/pop_total*100:.1f}%)")
    print(f"[COVERAGE] Population with delta = 0 : {pop_zero:>12,.0f} ({pop_zero/pop_total*100:.1f}%)")
    print(f"[COVERAGE] Population excluded (NaN) : {pop_excluded:>12,.0f} ({pop_excluded/pop_total*100:.1f}%)")
    print(f"\n[COVERAGE] Deaths with delta > 0     : {mort_covered:>12,.0f} ({mort_covered/mort_total*100:.1f}%)")
    print(f"[COVERAGE] Deaths excluded (NaN)     : {mort_excluded:>12,.0f} ({mort_excluded/mort_total*100:.1f}%)")

    covered_mask = comm_age["meandelta_comm"] > 0
    if covered_mask.any():
        natl_delta = np.average(
            comm_age.loc[covered_mask, "meandelta_comm"],
            weights=comm_age.loc[covered_mask, "pop_age"].clip(lower=1e-9)
        )
        RR_ref = float(RR_PM25_Hrapie if pol == "ug_PM25_RH50" else RR_NO2_Hrapie)
        print(f"\n[SANITY] Pop-weighted mean delta (covered): {natl_delta:.4f} µg/m³")
        print(f"[SANITY] Expected avoided deaths if fully covered: "
              f"{mort_total * (1.0 - np.exp(-np.log(RR_ref) * natl_delta / 10.0)):,.1f}")

    # --- Step 4: RR and central AF ---
    RR_dict_main = {
        "ug_PM25_RH50": (RR_PM25_Hrapie, RR_PM25_Hrapie_low, RR_PM25_Hrapie_high),
        "ug_NO2":       (RR_NO2_Hrapie,  RR_NO2_Hrapie_low,  RR_NO2_Hrapie_high),
    }
    RR, RR_low, RR_high = RR_dict_main[pol]
    RR, RR_low, RR_high = float(RR), float(RR_low), float(RR_high)

    lnRR_mean = np.log(RR)
    lnRR_sd   = max((np.log(RR_high) - np.log(RR_low)) / (2 * 1.96), 1e-12)
    sim_lnRRs = np.random.normal(lnRR_mean, lnRR_sd, size=num_simulations)

    delta_vals     = comm_age["meandelta_comm"].fillna(0).values
    comm_age["AF"] = 1.0 - np.exp(-lnRR_mean * delta_vals / 10.0)
    comm_age["mortpol"] = comm_age["morta_age"] * comm_age["AF"]

    # --- Step 5: aggregate to age level (all communes, for avoided deaths / YLG) ---
    comm_age["mort_x_delta"] = comm_age["morta_age"] * comm_age["meandelta_comm"].fillna(0)

    age_summary = (
        comm_age.groupby("age", as_index=False)
        .agg(
            pop_age      = ("pop_age",      "sum"),
            morta_age    = ("morta_age",    "sum"),
            mortpol_age  = ("mortpol",      "sum"),
            n_communes   = ("comcod",       "nunique"),
        )
    )

    # MC CIs for avoided deaths (all communes)
    age_eff_delta_all = np.nan_to_num(
        np.where(
            age_summary["morta_age"].values > 0,
            comm_age.groupby("age")["mort_x_delta"].sum().reindex(age_summary["age"]).values
            / age_summary["morta_age"].values,
            0.0
        ), nan=0.0
    )
    deaths_age      = age_summary["morta_age"].values[:, np.newaxis]
    delta_age_all   = age_eff_delta_all[:, np.newaxis]
    sim_avoided_age = deaths_age * (1.0 - np.exp(-sim_lnRRs[np.newaxis, :] * delta_age_all / 10.0))

    age_summary["mortpol_LCI"] = np.percentile(sim_avoided_age, 2.5,  axis=1)
    age_summary["mortpol_UCI"] = np.percentile(sim_avoided_age, 97.5, axis=1)

    # --- Step 6: LE rates and life table — EXPOSED POPULATION ONLY (delta > 0) ---
    comm_age_exposed = comm_age[comm_age["meandelta_comm"] > 0].copy()

    age_exposed = (
        comm_age_exposed.groupby("age", as_index=False)
        .agg(
            pop_age_exp   = ("pop_age",       "sum"),
            morta_age_exp = ("morta_age",     "sum"),
            mortpol_exp   = ("mortpol",       "sum"),
            mort_x_delta  = ("mort_x_delta",  "sum"),
        )
    )

    age_summary = age_summary.merge(age_exposed, on="age", how="left")
    age_summary[["pop_age_exp", "morta_age_exp", "mortpol_exp", "mort_x_delta"]] = \
        age_summary[["pop_age_exp", "morta_age_exp", "mortpol_exp", "mort_x_delta"]].fillna(0)

    # Mort-weighted effective delta for exposed pop — use nan_to_num, not .fillna (numpy array)
    age_summary["eff_delta_exp"] = np.nan_to_num(
        np.where(
            age_summary["morta_age_exp"].values > 0,
            age_summary["mort_x_delta"].values / age_summary["morta_age_exp"].values,
            0.0
        ), nan=0.0
    )

    # MC CIs for LE — exposed population only
    deaths_exp      = age_summary["morta_age_exp"].values[:, np.newaxis]
    delta_exp       = age_summary["eff_delta_exp"].values[:, np.newaxis]
    sim_avoided_exp = deaths_exp * (1.0 - np.exp(-sim_lnRRs[np.newaxis, :] * delta_exp / 10.0))

    mortpol_exp_LCI = np.percentile(sim_avoided_exp, 2.5,  axis=1)
    mortpol_exp_UCI = np.percentile(sim_avoided_exp, 97.5, axis=1)

    # Mortality rates for exposed population only
    pop_nz_exp = age_summary["pop_age_exp"].replace(0, np.nan)

    age_summary["taux_initial"]     = (age_summary["morta_age_exp"] / pop_nz_exp).fillna(0)
    age_summary["taux_corrige"]     = ((age_summary["morta_age_exp"] - age_summary["mortpol_exp"])  / pop_nz_exp).clip(0).fillna(0)
    age_summary["taux_corrige_LCI"] = ((age_summary["morta_age_exp"] - mortpol_exp_LCI)             / pop_nz_exp).clip(0, 1).fillna(0)
    age_summary["taux_corrige_UCI"] = ((age_summary["morta_age_exp"] - mortpol_exp_UCI)             / pop_nz_exp).clip(0, 1).fillna(0)

    # Life table on ages 30–99 directly; e_x[0] = remaining LE at age 30 (e_30)
    for rate_col, e30_col, ex_col in [
        ("taux_initial",     "e30_initial",     "LifeTable_LE_initial"),
        ("taux_corrige",     "e30_corrige",     "LifeTable_LE_corrige"),
        ("taux_corrige_LCI", "e30_corrige_LCI", "LifeTable_LE_corrige_LCI"),
        ("taux_corrige_UCI", "e30_corrige_UCI", "LifeTable_LE_corrige_UCI"),
    ]:
        e_x = life_expectancy_table(
            age_summary["age"].values,
            age_summary[rate_col].values,
            interval=1, a_x=0.5, radix=100000
        )
        age_summary[ex_col]  = e_x      # remaining LE at each age 30–99
        age_summary[e30_col] = e_x[0]   # scalar: remaining LE at age 30

    # --- e_30-based LE gain (scalar) ---
    overall_LEgain_mo     = (age_summary["e30_corrige"].iloc[0]     - age_summary["e30_initial"].iloc[0]) * 12
    overall_LEgain_mo_LCI = (age_summary["e30_corrige_LCI"].iloc[0] - age_summary["e30_initial"].iloc[0]) * 12
    overall_LEgain_mo_UCI = (age_summary["e30_corrige_UCI"].iloc[0] - age_summary["e30_initial"].iloc[0]) * 12

    # --- Pop-weighted remaining LE gain (ages 30–99, exposed pop) ---
    total_pop_exp = age_summary["pop_age_exp"].sum()
    if total_pop_exp > 0:
        pw_LE_initial     = np.sum(age_summary["LifeTable_LE_initial"]     * age_summary["pop_age_exp"]) / total_pop_exp
        pw_LE_corrige     = np.sum(age_summary["LifeTable_LE_corrige"]     * age_summary["pop_age_exp"]) / total_pop_exp
        pw_LE_corrige_LCI = np.sum(age_summary["LifeTable_LE_corrige_LCI"] * age_summary["pop_age_exp"]) / total_pop_exp
        pw_LE_corrige_UCI = np.sum(age_summary["LifeTable_LE_corrige_UCI"] * age_summary["pop_age_exp"]) / total_pop_exp
    else:
        pw_LE_initial = pw_LE_corrige = pw_LE_corrige_LCI = pw_LE_corrige_UCI = 0.0

    pw_LEgain_mo     = (pw_LE_corrige     - pw_LE_initial) * 12
    pw_LEgain_mo_LCI = (pw_LE_corrige_LCI - pw_LE_initial) * 12
    pw_LEgain_mo_UCI = (pw_LE_corrige_UCI - pw_LE_initial) * 12

    # --- YLG (uses all-commune mortpol but exposed-pop LE for remaining life) ---
    age_summary["YLG"]     = age_summary["LifeTable_LE_initial"] * age_summary["mortpol_age"]
    age_summary["YLG_LCI"] = age_summary["LifeTable_LE_initial"] * age_summary["mortpol_LCI"]
    age_summary["YLG_UCI"] = age_summary["LifeTable_LE_initial"] * age_summary["mortpol_UCI"]

    total_avoided = age_summary["mortpol_age"].sum()
    total_YLG     = age_summary["YLG"].sum()

    age_summary["overall_avoided_deaths"]        = total_avoided
    age_summary["overall_avoided_deaths_LCI"]    = age_summary["mortpol_LCI"].sum()
    age_summary["overall_avoided_deaths_UCI"]    = age_summary["mortpol_UCI"].sum()
    age_summary["overall_YLG"]                   = total_YLG
    age_summary["overall_YLG_LCI"]               = age_summary["YLG_LCI"].sum()
    age_summary["overall_YLG_UCI"]               = age_summary["YLG_UCI"].sum()
    age_summary["overall_YLG_per_avoided_death"] = total_YLG / total_avoided if total_avoided > 0 else 0.0

    age_summary["overall_LifeTable_LEgain_mo"]     = overall_LEgain_mo
    age_summary["overall_LifeTable_LEgain_mo_LCI"] = overall_LEgain_mo_LCI
    age_summary["overall_LifeTable_LEgain_mo_UCI"] = overall_LEgain_mo_UCI

    age_summary["overall_pw_LEgain_mo"]     = pw_LEgain_mo
    age_summary["overall_pw_LEgain_mo_LCI"] = pw_LEgain_mo_LCI
    age_summary["overall_pw_LEgain_mo_UCI"] = pw_LEgain_mo_UCI

    print(f"\n[RESULT] Total avoided deaths       : {total_avoided:>12,.1f}")
    print(f"[RESULT] Total YLG                  : {total_YLG:>12,.1f}")
    print(f"[RESULT] LE gain e30 (mo)           : {overall_LEgain_mo:>12,.2f}  [{overall_LEgain_mo_LCI:.2f}, {overall_LEgain_mo_UCI:.2f}]")
    print(f"[RESULT] LE gain pop-weighted (mo)  : {pw_LEgain_mo:>12,.2f}  [{pw_LEgain_mo_LCI:.2f}, {pw_LEgain_mo_UCI:.2f}]")

    return age_summary.sort_values("age").reset_index(drop=True)

In [6]:
# Function to run mortality avoided and life expectancy analysis with MC simulation
# and at population-weighted commune concentration — running ALL clusters in a loop
import os
import logging
import warnings
import pandas as pd

warnings.simplefilter(action="ignore", category=FutureWarning)

logging.basicConfig(level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s")

# Define scenarios, pollutants, and years
scenarios = ["s1", "s2", "s3", "s4"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]  # run one-by-one to avoid memory issues
years = ["2030", "2050"]

# --------- CLUSTER FILTER UTILS ---------
def filter_by_cluster(donnees_exportees_transformed, donnees_merged, clusters):
    """
    Filters both:
      - donnees_exportees_transformed (used by expo)
      - donnees_merged (used by mortalite_age_commune_monte_carlo)

    Assumes 'comcod' exists in both.
    Assumes area_cluster exists in donnees_exportees_transformed; if missing in donnees_merged,
    it will be merged in from donnees_exportees_transformed.
    """
    clusters = [c.strip() for c in clusters]

    # Make sure comcod is comparable
    donnees_exportees_transformed = donnees_exportees_transformed.copy()
    donnees_merged = donnees_merged.copy()
    donnees_exportees_transformed["comcod"] = donnees_exportees_transformed["comcod"].astype(str).str.zfill(5).str.strip()
    donnees_merged["comcod"] = donnees_merged["comcod"].astype(str).str.zfill(5).str.strip()

    # If needed, attach area_cluster to donnees_merged from IRIS/commune data
    if "area_cluster" not in donnees_merged.columns:
        lut = (donnees_exportees_transformed[["comcod", "area_cluster"]]
               .dropna(subset=["comcod"])
               .drop_duplicates())
        donnees_merged = donnees_merged.merge(lut, on="comcod", how="left")

    # Filter
    de_f = donnees_exportees_transformed.loc[
        donnees_exportees_transformed["area_cluster"].isin(clusters)
    ].copy()

    dm_f = donnees_merged.loc[
        donnees_merged["area_cluster"].isin(clusters)
    ].copy()

    # Enforce intersection of communes so both datasets cover the same set
    common = set(de_f["comcod"]).intersection(set(dm_f["comcod"]))
    de_f = de_f[de_f["comcod"].isin(common)].copy()
    dm_f = dm_f[dm_f["comcod"].isin(common)].copy()

    return de_f, dm_f

def safe_cluster_slug(clusters):
    """Turn ['Semi-urban','Urban dense'] into a filesystem-safe slug."""
    slug = "_".join(clusters)
    slug = slug.replace(" ", "-").replace("/", "-")
    return slug

# --------- CORE PIPELINE ---------
def process_combination(params, donnees_exportees_transformed, grille_combinee, donnees_merged, cluster_slug):
    scenario, year, pollutant = params
    logging.info(
        f"Processing: cluster={cluster_slug} | Scenario={scenario} | Pollutant={pollutant} | Year={year}"
    )

    try:
        # Load computation data
        logging.info("Starting coordo_chimere function")
        conc_points = coordo_chimere(pollutant, year=year, SC=scenario.upper())
        conc_points = conc_points.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished coordo_chimere")

        logging.info("Starting coordo_ineris function")
        conc_chimere = coordo_ineris_chimere(pollutant, year="2019")
        conc_chimere = conc_chimere.to_crs(donnees_exportees_transformed.crs)
        logging.info("Finished coordo_ineris_chimere")

        # Apply correction for concentrations
        conc_corrige = correction_chimere(conc_points, conc_chimere)
        conc_corrige = conc_corrige.to_crs(donnees_exportees_transformed.crs)

        # Exposure calculation (on filtered geometries)
        donnees_expo = expo_fast(donnees_exportees_transformed,conc_corrige,grille_combinee,max_distance_m=None,fallback_for_missing=True)
        logging.info("Exposure processing completed")
        # HIA computation (on filtered mortality/population table)
        grouped = mortalite_age_commune_monte_carlo(
            mort_comm_age=donnees_merged,
            donnees_transformed=donnees_exportees_transformed,
            donnees_expo=donnees_expo,
            annee=year,
            pol=pollutant,
            num_simulations=1000,
        )

        # Save result
        output_path = os.path.join("data", "2-output-data", scenario, pollutant, year, cluster_slug)
        os.makedirs(output_path, exist_ok=True)
        out_csv = os.path.join(output_path, "mortality_chimere_clusters.csv")
        grouped.to_csv(out_csv, index=False)
        logging.info(f"Exported: {out_csv}")

    except Exception as e:
        logging.error(f"Error: cluster={cluster_slug} | Scenario={scenario} | Pollutant={pollutant} | Year={year}")
        logging.exception(f"Error details: {e}")

# --------- MAIN ---------
if __name__ == "__main__":
    # Align CRS of shapefiles to the CRS of donnees_exportees_transformed
    target_crs = donnees_exportees_transformed.crs
    donnees_exportees_transformed = donnees_exportees_transformed.to_crs(target_crs)
    grille_combinee = grille_combinee.to_crs(target_crs)

    # OPTIONAL: ensure comcod format early
    donnees_exportees_transformed["comcod"] = donnees_exportees_transformed["comcod"].astype(str).str.zfill(5).str.strip()
    donnees_merged["comcod"] = donnees_merged["comcod"].astype(str).str.zfill(5).str.strip()

    # Define which clusters to run (edit as needed)
    clusters_to_run_list = [
        ["Rural"],
        ["Peri-Urban"],
        ["Urban"],
        # If you also want unmatched/NA:
        # ["NA"],
        # Or combinations:
        # ["Semi-urban", "Urban dense"],
    ]

    # Prepare parameters for processing
    params_list = [(scenario, year, pollutant) for scenario in scenarios for year in years for pollutant in pollutants]

    # Run loop: cluster -> scenario/year/pollutant
    for clusters_to_run in clusters_to_run_list:
        cluster_slug = safe_cluster_slug(clusters_to_run)
        de_f, dm_f = filter_by_cluster(donnees_exportees_transformed, donnees_merged, clusters_to_run)
        logging.info(f"========== RUNNING CLUSTERS {clusters_to_run} (slug={cluster_slug}) ==========")
        logging.info(f"Kept IRIS rows: {len(de_f)} | Kept mortality rows: {len(dm_f)}")
        logging.info(f"Unique communes kept: {de_f['comcod'].nunique()}")

        if len(de_f) == 0 or len(dm_f) == 0:
            logging.warning(f"Skipping cluster {clusters_to_run}: empty filtered dataset.")
            continue

        for params in params_list:
            process_combination(params, de_f, grille_combinee, dm_f, cluster_slug)

    logging.info("Successfully processed all combinations for all clusters.")

INFO:root:========== RUNNING CLUSTERS ['Rural'] (slug=Rural) ==========
INFO:root:Kept IRIS rows: 30119 | Kept mortality rows: 3011900
INFO:root:Unique communes kept: 29693
INFO:root:Processing: cluster=Rural | Scenario=s1 | Pollutant=ug_PM25_RH50 | Year=2030
INFO:root:Starting coordo_chimere function


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS1_FRA02_PM25_analysis_yravg.nc


INFO:root:Finished coordo_chimere
INFO:root:Starting coordo_ineris function


Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc


INFO:root:Finished coordo_ineris_chimere


...

Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc


INFO:root:Finished coordo_ineris_chimere


Finished processing coordo_ineris function


INFO:root:Starting expo_fast
INFO:root:expo_fast completed
INFO:root:Exposure processing completed
INFO:root:Exported: data\2-output-data\s4\ug_NO2\2050\Urban\mortality_chimere_clusters.csv
INFO:root:Successfully processed all combinations for all clusters.


[COVERAGE] Communes with delta > 0   : 759
[COVERAGE] Communes with delta = 0   : 17
[COVERAGE] Communes with delta NaN   : 0 (excluded)

[COVERAGE] Population with delta > 0 :   16,164,063 (99.7%)
[COVERAGE] Population with delta = 0 :       43,341 (0.3%)
[COVERAGE] Population excluded (NaN) :            0 (0.0%)

[COVERAGE] Deaths with delta > 0     :      287,231 (99.7%)
[COVERAGE] Deaths excluded (NaN)     :            0 (0.0%)

[SANITY] Pop-weighted mean delta (covered): 6.8112 µg/m³
[SANITY] Expected avoided deaths if fully covered: 9,413.1

[RESULT] Total avoided deaths       :      9,533.6
[RESULT] Total YLG                  :     72,206.2
[RESULT] LE gain e30 (mo)           :         3.18  [2.01, 4.60]
[RESULT] LE gain pop-weighted (mo)  :         2.61  [1.69, 3.81]


In [7]:
import os
import glob
import pandas as pd

BASE_DIR = os.path.join("data", "2-output-data")
FILENAME = "mortality_chimere_clusters.csv"

def pick_col(df, candidates):
    """Return the first existing column from candidates, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def find_cols(df):
    """
    Map required outputs to column names in your output file.
    Edit candidates if your file uses different naming.
    """
    col_avoided     = pick_col(df, ["overall_avoided_deaths", "avoided_deaths", "overall_mortpol", "mort_avoided"])
    col_avoided_lci = pick_col(df, ["overall_avoided_deaths_LCI", "avoided_deaths_LCI", "overall_mortpol_LCI"])
    col_avoided_uci = pick_col(df, ["overall_avoided_deaths_UCI", "avoided_deaths_UCI", "overall_mortpol_UCI"])

    # LE gain at age 30 (often in months)
    col_age         = pick_col(df, ["age", "Age", "age_class"])
    col_LEgain_mo   = pick_col(df, ["LifeTable_LEgain_mo", "LEgain_mo", "LifeTable_LE_gain_mo"])
    col_LEgain_lci  = pick_col(df, ["LifeTable_LEgain_mo_LCI", "LEgain_mo_LCI"])
    col_LEgain_uci  = pick_col(df, ["LifeTable_LEgain_mo_UCI", "LEgain_mo_UCI"])

    # Pop-weighted overall LE gain (months)
    col_popw        = pick_col(df, ["overall_LifeTable_LEgain_mo_popw", "overall_LifeTable_LEgain_mo", "overall_LEgain_mo_popw"])
    col_popw_lci    = pick_col(df, ["overall_LifeTable_LEgain_mo_popw_LCI", "overall_LifeTable_LEgain_mo_LCI"])
    col_popw_uci    = pick_col(df, ["overall_LifeTable_LEgain_mo_popw_UCI", "overall_LifeTable_LEgain_mo_UCI"])

    return {
        "avoided": (col_avoided, col_avoided_lci, col_avoided_uci),
        "age": col_age,
        "LEgain": (col_LEgain_mo, col_LEgain_lci, col_LEgain_uci),
        "LEpopw": (col_popw, col_popw_lci, col_popw_uci),
    }

def safe_div(num, denom):
    if num is None or denom is None or pd.isna(num) or pd.isna(denom) or denom == 0:
        return None
    return float(num) / float(denom)

def extract_one_file(path, scenario, pollutant, year, cluster):
    df = pd.read_csv(path)
    cols = find_cols(df)

    # ---- overall avoided deaths ----
    a, a_lci, a_uci = cols["avoided"]
    if a is None:
        raise KeyError(f"[{path}] Could not find overall avoided deaths column. Available: {list(df.columns)}")

    avoided     = float(df[a].iloc[0])
    avoided_lci = float(df[a_lci].iloc[0]) if a_lci else None
    avoided_uci = float(df[a_uci].iloc[0]) if a_uci else None

    # ---- LE gain at age 30 ----
    age_col = cols["age"]
    lg, lg_lci, lg_uci = cols["LEgain"]

    le30 = le30_lci = le30_uci = None
    if age_col and lg:
        age_num = pd.to_numeric(df[age_col], errors="coerce")
        row30 = df.loc[age_num == 30]
        if not row30.empty:
            le30     = float(row30[lg].iloc[0])
            le30_lci = float(row30[lg_lci].iloc[0]) if lg_lci else None
            le30_uci = float(row30[lg_uci].iloc[0]) if lg_uci else None

    # ---- Pop-weighted overall LE gain ----
    pw, pw_lci, pw_uci = cols["LEpopw"]
    popw = popw_lci = popw_uci = None
    if pw:
        popw     = float(df[pw].iloc[0])
        popw_lci = float(df[pw_lci].iloc[0]) if pw_lci else None
        popw_uci = float(df[pw_uci].iloc[0]) if pw_uci else None

    return {
        "scenario": scenario,
        "year": year,
        "pollutant": pollutant,
        "cluster": cluster,

        "avoided_deaths": avoided,
        "avoided_deaths_LCI": avoided_lci,
        "avoided_deaths_UCI": avoided_uci,

        "LE_gain_age30_mo": le30,
        "LE_gain_age30_mo_LCI": le30_lci,
        "LE_gain_age30_mo_UCI": le30_uci,

        "LE_gain_popw_mo": popw,
        "LE_gain_popw_mo_LCI": popw_lci,
        "LE_gain_popw_mo_UCI": popw_uci,
    }

def add_cluster_shares(df):
    group_cols = ["scenario", "pollutant", "year"]

    total_avoided = df.groupby(group_cols)["avoided_deaths"].transform("sum")
    df["fraction_mortality_avoided"] = df["avoided_deaths"] / total_avoided
    df["percent_mortality_avoided"] = 100.0 * df["fraction_mortality_avoided"]

    if "avoided_deaths_LCI" in df.columns:
        total_avoided_lci = df.groupby(group_cols)["avoided_deaths_LCI"].transform("sum")
        df["fraction_mortality_avoided_LCI"] = df["avoided_deaths_LCI"] / total_avoided_lci
        df["percent_mortality_avoided_LCI"] = 100.0 * df["fraction_mortality_avoided_LCI"]

    if "avoided_deaths_UCI" in df.columns:
        total_avoided_uci = df.groupby(group_cols)["avoided_deaths_UCI"].transform("sum")
        df["fraction_mortality_avoided_UCI"] = df["avoided_deaths_UCI"] / total_avoided_uci
        df["percent_mortality_avoided_UCI"] = 100.0 * df["fraction_mortality_avoided_UCI"]

    return df

def collect_all(base_dir=BASE_DIR):
    rows = []

    # data/2-output-data/<scenario>/<pollutant>/<year>/<cluster>/mortality_chimere_clusters.csv
    pattern = os.path.join(base_dir, "*", "*", "*", "*", FILENAME)
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError(f"No files found with pattern: {pattern}")

    for path in files:
        rel = os.path.relpath(path, base_dir)
        parts = rel.split(os.sep)
        # parts should be: [scenario, pollutant, year, cluster, filename]
        if len(parts) < 5:
            continue

        scenario, pollutant, year, cluster = parts[0], parts[1], parts[2], parts[3]

        try:
            rows.append(extract_one_file(path, scenario, pollutant, year, cluster))
        except Exception as e:
            print(f"Skipping {path}: {e}")

    out = pd.DataFrame(rows)

    if out.empty:
        return out

    out = add_cluster_shares(out)

    # nice ordering for publication
    out = out.sort_values(["pollutant", "scenario", "year", "cluster"]).reset_index(drop=True)
    return out

if __name__ == "__main__":
    pub_table = collect_all(BASE_DIR)

    out_csv = os.path.join(BASE_DIR, "publication_table_mort_LE.csv")
    pub_table.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
    print(pub_table.head(20))

Saved: data\2-output-data\publication_table_mort_LE.csv
   scenario  year pollutant     cluster  avoided_deaths  avoided_deaths_LCI  \
0        s1  2030    ug_NO2  Peri-Urban     5389.037651         3462.943514   
1        s1  2030    ug_NO2       Rural     4245.597341         2735.915850   
2        s1  2030    ug_NO2       Urban     5724.475934         3635.969344   
3        s1  2050    ug_NO2  Peri-Urban     9604.420841         5715.890059   
4        s1  2050    ug_NO2       Rural     7587.569389         4365.674816   
5        s1  2050    ug_NO2       Urban    10244.419391         6142.356522   
6        s2  2030    ug_NO2  Peri-Urban     5273.618338         3262.729750   
7        s2  2030    ug_NO2       Rural     4151.270073         2569.284001   
8        s2  2030    ug_NO2       Urban     5594.520925         3429.443011   
9        s2  2050    ug_NO2  Peri-Urban     9587.072676         5648.824530   
10       s2  2050    ug_NO2       Rural     7569.206110         4689.045459